# Train DeepLabV3+ (ResNet50 + SSL) บนชุดข้อมูล CULane

สมุดงานนี้กำหนดค่าไว้เฉพาะสำหรับ:
- **Architecture:** `DeepLabV3Plus`
- **Encoder (Backbone):** `resnet50`
- **Pre-trained Weights:** `ssl` (Semi-supervised Learning / MoCo v2)
- **Hyperparameters:** `EPOCHS = 30`, `BATCH_SIZE = 16`, `LR = 1e-4`

เมตริกวัดผล: **IoU, Precision, Recall, F1-Score** (ไม่มีเมตริกมุมความชัน slope)

In [ ]:
!pip install opencv-python matplotlib numpy segmentation-models-pytorch tqdm kornia

In [ ]:
import os

# ต้องตั้งก่อน import torch — ช่วยลดปัญหา memory fragmentation ตอนใช้ batch size ใหญ่
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ให้แน่ใจว่า cwd อยู่ที่โฟลเดอร์ cardashcam_train_model เพื่อให้ path models/ และ results/ ถูกต้อง
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import sys
import csv

if sys.stdout and hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

import cv2

# ปิด thread ภายในของ OpenCV
# เพราะ DataLoader จะแตก worker หลาย process อยู่แล้ว ถ้าปล่อยให้ cv2 แตก thread เองอีก
# จะแย่ง CPU กันเอง (container นี้ถูกจำกัดไว้แค่ 4 cores)
cv2.setNumThreads(0)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm
import segmentation_models_pytorch as smp

import numpy as np     # เซลล์นี้เดิมไม่ได้ import numpy — ตอนนี้ลูป val ต้องใช้

sys.path.append('.')  # ให้ Python เห็นไฟล์ dataset.py ในโฟลเดอร์เดียวกัน
from dataset import LaneDatasetFromCSV, IMG_SIZE


print("import สำเร็จ")
print(f"torch {torch.__version__} | CUDA พร้อมใช้: {torch.cuda.is_available()}")
print(f"smp {smp.__version__} | IMG_SIZE ที่ dataset.py ใช้: {IMG_SIZE} (สูง, กว้าง)")
print(f"working directory: {os.getcwd()}")


## ฟังก์ชัน metric (IoU, Precision, Recall, F1)

ใช้สูตรเดียวกับ `train_unet_model.ipynb` เป๊ะๆ เพื่อให้ตัวเลขเทียบกับผลการทดลองเก่าใน `results/` ได้

In [ ]:
def calculate_iou(outputs, labels, threshold=0.5):
    preds = torch.sigmoid(outputs) > threshold
    labels = labels > threshold
    intersection = (preds & labels).float().sum((1, 2, 3))
    union = (preds | labels).float().sum((1, 2, 3))
    return ((intersection + 1e-6) / (union + 1e-6)).mean().item()


def calculate_precision_recall_f1(outputs, labels, threshold=0.5, eps=1e-6):
    preds = torch.sigmoid(outputs) > threshold
    labels = labels > threshold
    tp = (preds & labels).float().sum((1, 2, 3))
    fp = (preds & ~labels).float().sum((1, 2, 3))
    fn = (~preds & labels).float().sum((1, 2, 3))
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    f1 = (2 * precision * recall + eps) / (precision + recall + eps)
    return precision.mean().item(), recall.mean().item(), f1.mean().item()

print("ฟังก์ชัน metric พร้อมแล้ว")

## Loss function (BCE + Dice)

⚠️ **จุดสำคัญ:** `DiceLoss` ต้อง cast เป็น `float()` (fp32) ก่อนคำนวณ

ตอนใช้ AMP ค่า output จะเป็น fp16 การ `.sum()` ค่าประมาณ 11.5 ล้านตัว (32 × 448 × 800)
จะได้ผลรวมราว 1.1e7 ซึ่ง **ทะลุเพดานของ fp16 (65504)** แล้วกลายเป็น `inf` → loss เป็น NaN
เทรนไม่ขึ้นเลย

ทดสอบบนเครื่องนี้แล้วเกิดขึ้นจริง: แบบเดิม `.sum()` ได้ `inf` แต่แบบ upcast ได้ `10924882.0`
ส่วน `BCEWithLogitsLoss` ปลอดภัยกับ AMP อยู่แล้ว

In [ ]:
class DiceLoss(nn.Module):
    def forward(self, inputs, targets, smooth=1):
        # .float() คือหัวใจ: กัน fp16 overflow ตอนรวมค่า (ดูคำอธิบายในเซลล์ markdown ด้านบน)
        inputs = torch.sigmoid(inputs.float()).view(-1)
        targets = targets.float().view(-1)
        intersection = (inputs * targets).sum()
        dice = (2. * intersection + smooth) / (inputs.sum() + targets.sum() + smooth)
        return 1 - dice

print("DiceLoss พร้อมแล้ว (upcast fp32 กัน overflow ตอนใช้ AMP)")

## เตรียม input: ImageNet normalization + channels_last

**1) normalization** — `dataset.py` ส่งรูปออกมาเป็นช่วง `[0, 1]` เฉยๆ (หาร 255) **ไม่มี** การ
normalize ด้วย mean/std แต่ encoder ที่ pretrain จาก ImageNet คาดหวัง input ที่ normalize แล้ว
เราจึงทำ **ในนี้** (ไม่ไปแก้ `dataset.py`) โดยทำ **หลัง** ย้ายขึ้น GPU และ **หลัง** augmentation
ของ dataset (ซึ่ง clamp ค่าไว้ใน `[0, 1]` แล้ว) — **ใช้กับรูปเท่านั้น ห้ามใช้กับ mask**

**2) channels_last** — เปลี่ยน memory layout ให้ conv ใช้ tensor core ได้เต็มที่
ผลลัพธ์เหมือนเดิมทางคณิตศาสตร์ แต่วัดบนเครื่องนี้ได้ **เร็วขึ้น 1.46 เท่า**
(0.535 → 0.366 วินาที/step ทำให้ 30 epochs ลดจาก ~8.5 ชม. เหลือ ~5.8 ชม.)

ทั้งสองอย่างต้องทำเหมือนกันทั้ง train / val / test จึงรวมไว้ในฟังก์ชันเดียว

In [ ]:
def get_norm_stats(encoder_name, encoder_weights):
    """ดึงค่า mean/std ที่ encoder ตัวนั้น pretrain มา ถ้าดึงไม่ได้ก็ใช้ค่ามาตรฐาน ImageNet"""
    mean = (0.485, 0.456, 0.406)
    std = (0.229, 0.224, 0.225)
    try:
        params = smp.encoders.get_preprocessing_params(encoder_name, pretrained=encoder_weights)
        if params.get('mean'):
            mean = tuple(params['mean'])
        if params.get('std'):
            std = tuple(params['std'])
    except Exception as e:
        print(f"⚠️ ดึง preprocessing params จาก smp ไม่ได้ ({e}) — ใช้ค่ามาตรฐาน ImageNet แทน")
    return mean, std


def make_input_prep(encoder_name, encoder_weights, device,
                    normalize=True, channels_last=False):
    """
    คืนฟังก์ชันเตรียม input สำหรับรูป (batch, 3, H, W) — ใช้กับรูปเท่านั้น ไม่ใช้กับ mask

    normalize=False ใช้ตอนประเมินโมเดลเก่าที่เทรนมาด้วย input [0,1] แบบไม่ normalize
    """
    mean_t = std_t = None
    if normalize:
        mean, std = get_norm_stats(encoder_name, encoder_weights)
        mean_t = torch.tensor(mean, device=device).view(1, 3, 1, 1)
        std_t = torch.tensor(std, device=device).view(1, 3, 1, 1)
        print(f"🎨 normalize ด้วย mean={mean}, std={std}")
    else:
        print("🎨 ไม่ normalize (ใช้ input ช่วง [0,1] ตรงๆ)")

    print(f"🧭 memory format: {'channels_last' if channels_last else 'contiguous (ปกติ)'}")

    def prep(images):
        if mean_t is not None:
            images = (images - mean_t) / std_t
        if channels_last:
            images = images.contiguous(memory_format=torch.channels_last)
        return images

    return prep

print("ฟังก์ชันเตรียม input พร้อมแล้ว")

## ฟังก์ชันวาดกราฟ

In [ ]:
def save_plot(history_train_loss, history_val_loss,
              history_train_iou, history_val_iou,
              history_train_recall, history_val_recall,
              history_train_f1, history_val_f1,
              graph_path):
    plt.figure(figsize=(14, 10))

    plt.subplot(2, 2, 1)
    plt.plot(history_train_loss, label='Train Loss', color='red')
    plt.plot(history_val_loss, label='Val Loss', color='orange')
    plt.title('Loss History', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.subplot(2, 2, 2)
    plt.plot(history_train_iou, label='Train Accuracy (IoU)', color='green')
    plt.plot(history_val_iou, label='Val Accuracy (IoU)', color='blue')
    plt.title('Accuracy (IoU) History', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch')
    plt.ylabel('IoU')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.subplot(2, 2, 3)
    plt.plot(history_train_recall, label='Train Recall', color='purple')
    plt.plot(history_val_recall, label='Val Recall', color='magenta')
    plt.title('Recall History', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch')
    plt.ylabel('Recall')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.subplot(2, 2, 4)
    plt.plot(history_train_f1, label='Train F1-score', color='teal')
    plt.plot(history_val_f1, label='Val F1-score', color='navy')
    plt.title('F1-score History', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch')
    plt.ylabel('F1')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(graph_path, dpi=110)
    plt.close()

print("ฟังก์ชันวาดกราฟพร้อมแล้ว (4 เมตริก: Loss, IoU, Recall, F1)")


## ฟังก์ชันเทรนหลัก

รับค่าจาก CONFIG ด้านล่าง ไม่ต้องแก้โค้ดในเซลล์นี้

**ไฟล์ที่จะได้ออกมา** (ชื่อมาจาก `exp_name = {encoder}_{weights}_bs{batch}_ep{epochs}`)

| ไฟล์ | คืออะไร |
| --- | --- |
| `models/final_result_epoch_{exp_name}.pth` | epoch สุดท้าย (มี history + optimizer + scaler) ใช้ resume |
| `models/best_model_{exp_name}.pth` | ตัวที่ `val_loss` ต่ำสุด |
| `models/unet_lane_model_{exp_name}.pth` | โมเดลตอนเทรนจบ (epoch สุดท้าย) — เซลล์ test ใช้ตัวนี้ |
| `results/metrics_{exp_name}.csv` | ตัวเลขทุก epoch |
| `results/training_history_{exp_name}.png` | กราฟ |

**เรื่อง resume:** ถ้ามี checkpoint อยู่แล้วและ `epoch >= epochs` ฟังก์ชันจะ **ข้ามการเทรนทันที**
ถ้าอยากเทรนใหม่ตั้งแต่ต้นต้องลบ `models/final_result_epoch_{exp_name}.pth` ก่อน
หรือถ้าอยากเทรนต่อให้เพิ่มค่า `EPOCHS`

In [ ]:
def _subset_indices(total, n):
    """
    เลือก n index กระจายทั่วทั้งชุดแบบ stride (หยิบทุกๆ k ตัว) คืน None ถ้าไม่ต้องจำกัด
    """
    if not n or n >= total:
        return None
    return [int(i * total / n) for i in range(n)]


def _limited(dataset, n):
    """ห่อ dataset ด้วย Subset ถ้ามีการจำกัดจำนวน คืน (dataset, index หรือ None)"""
    idx = _subset_indices(len(dataset), n)
    if idx is None:
        return dataset, None
    return torch.utils.data.Subset(dataset, idx), idx


def _cat_counts(base_ds, idx=None):
    """นับจำนวนรูปต่อ category"""
    if idx is None:
        return base_ds.category_counts()
    counts = {}
    for i in idx:
        c = base_ds.samples[i][1]
        counts[c] = counts.get(c, 0) + 1
    return dict(sorted(counts.items()))


def train(encoder_name, encoder_weights, epochs, batch_size, lr, dataset_root,
          num_workers=4, use_amp=True, use_normalize=True, use_channels_last=True,
          limit_train=None, limit_val=None, arch="DeepLabV3Plus"):
    arch_prefix = "deeplabv3plus_" if arch.lower() in ("deeplabv3plus", "deeplabv3+") else ""
    exp_name = f"{arch_prefix}{encoder_name}_{encoder_weights}_bs{batch_size}_ep{epochs}"
    if limit_train:
        exp_name += f"_n{limit_train}"

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = use_amp and device.type == "cuda"
    use_channels_last = use_channels_last and device.type == "cuda"

    print(f"🚀 ชิปประมวลผล: {device.type.upper()}")
    print(f"🧠 การทดลอง: arch={arch}, encoder={encoder_name}, weights={encoder_weights}, "
          f"epochs={epochs}, batch_size={batch_size}, lr={lr}")
    print(f"⚡ AMP: {'เปิด' if use_amp else 'ปิด'} | num_workers: {num_workers}")

    img_dir = os.path.join(dataset_root, "images")
    mask_dir = os.path.join(dataset_root, "masks")
    train_csv = os.path.join(dataset_root, "train_list.csv")
    val_csv = os.path.join(dataset_root, "val_list.csv")

    for path in [img_dir, mask_dir, train_csv, val_csv]:
        if not os.path.exists(path):
            print(f"❌ ไม่พบ path ที่จำเป็น: {path}")
            return

    base_train = LaneDatasetFromCSV(train_csv, img_dir, mask_dir, augment=True)
    base_val = LaneDatasetFromCSV(val_csv, img_dir, mask_dir, augment=False)
    train_dataset, train_idx = _limited(base_train, limit_train)
    val_dataset, val_idx = _limited(base_val, limit_val)

    loader_kwargs = dict(num_workers=num_workers, pin_memory=(device.type == "cuda"))
    if num_workers > 0:
        loader_kwargs.update(persistent_workers=True, prefetch_factor=2)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, **loader_kwargs)

    if train_idx is not None or val_idx is not None:
        print(f"✂️  เทรนแบบ subset ไม่ใช่ข้อมูลเต็ม — ไฟล์ผลลัพธ์จะใช้ชื่อ {exp_name}")
    print(f"📚 Train: {len(train_dataset)} / {len(base_train)} รูป {_cat_counts(base_train, train_idx)}")
    print(f"   Val:   {len(val_dataset)} / {len(base_val)} รูป {_cat_counts(base_val, val_idx)}")

    arch_fn = getattr(smp, arch, None)
    if arch_fn is None:
        raise ValueError(f"ไม่รู้จัก architecture: {arch} ใน segmentation_models_pytorch")
    model = arch_fn(encoder_name=encoder_name, encoder_weights=encoder_weights,
                    in_channels=3, classes=1).to(device)
    if use_channels_last:
        model = model.to(memory_format=torch.channels_last)

    prep_input = make_input_prep(encoder_name, encoder_weights, device,
                                 normalize=use_normalize, channels_last=use_channels_last)

    bce_criterion = nn.BCEWithLogitsLoss()
    dice_criterion = DiceLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    os.makedirs("models", exist_ok=True)
    os.makedirs("results/training", exist_ok=True)
    latest_ckpt_path = f"models/final_result_epoch_{exp_name}.pth"
    best_ckpt_path = f"models/best_model_{exp_name}.pth"
    model_prefix = "unet_lane_model" if arch.lower() == "unet" else "final_model"
    final_model_path = f"models/{model_prefix}_{exp_name}.pth"
    csv_path = f"results/training/metrics_{exp_name}.csv"
    graph_path = f"results/training/training_history_{exp_name}.png"

    history_train_loss, history_train_iou = [], []
    history_train_recall, history_train_f1 = [], []
    history_val_loss, history_val_iou = [], []
    history_val_recall, history_val_f1 = [], []
    best_val_loss = float('inf')
    best_epoch = 0
    start_epoch = 0

    print(f"\n🎓 เริ่มการทดลอง: {exp_name}")

    if os.path.exists(latest_ckpt_path):
        ckpt = torch.load(latest_ckpt_path, map_location=device, weights_only=False)
        if ckpt['epoch'] < epochs:
            print("🔄 พบไฟล์ที่เทรนค้างไว้! resume ต่อ")
            model.load_state_dict(ckpt['model_state_dict'])
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            start_epoch = ckpt['epoch']
            best_val_loss = ckpt.get('best_val_loss', float('inf'))
            best_epoch = ckpt.get('best_epoch', 0)
            history_train_loss = ckpt.get('history_train_loss', [])
            history_train_iou = ckpt.get('history_train_iou', [])
            history_train_recall = ckpt.get('history_train_recall', [])
            history_train_f1 = ckpt.get('history_train_f1', [])
            history_val_loss = ckpt.get('history_val_loss', [])
            history_val_iou = ckpt.get('history_val_iou', [])
            history_val_recall = ckpt.get('history_val_recall', [])
            history_val_f1 = ckpt.get('history_val_f1', [])
            if use_amp and ckpt.get('scaler_state_dict') and scaler:
                scaler.load_state_dict(ckpt['scaler_state_dict'])
            print(f"   resume ที่ epoch {start_epoch+1}/{epochs}")
        elif ckpt['epoch'] >= epochs:
            print(f"✅ ไฟล์นี้เทรนครบ {epochs} epochs แล้ว ข้ามการเทรน")
            return

    if start_epoch == 0:
        with open(csv_path, "w", newline="") as f:
            csv.writer(f).writerow(["epoch", "train_loss", "train_iou",
                                    "train_recall", "train_f1",
                                    "val_loss", "val_iou",
                                    "val_recall", "val_f1"])

    for epoch in range(start_epoch, epochs):
        model.train()
        running_loss = running_iou = running_recall = running_f1 = 0.0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for images, masks in pbar:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            images = prep_input(images)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=use_amp):
                outputs = model(images)
                outputs_f = outputs.float()
                loss = bce_criterion(outputs_f, masks) + dice_criterion(outputs_f, masks)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            with torch.no_grad():
                running_iou += calculate_iou(outputs_f, masks)
                _, r, f1 = calculate_precision_recall_f1(outputs_f, masks)

            running_loss += loss.item()
            running_recall += r
            running_f1 += f1
            pbar.set_postfix({"loss": f"{loss.item():.4f}", "iou": f"{running_iou/len(pbar):.4f}", "f1": f"{f1:.4f}"})

        n = len(train_loader)
        epoch_train_loss = running_loss / n
        epoch_train_iou = running_iou / n
        epoch_train_recall = running_recall / n
        epoch_train_f1 = running_f1 / n

        model.eval()
        val_loss = val_iou = val_recall = val_f1 = 0.0

        with torch.inference_mode():
            for images, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False):
                images = images.to(device, non_blocking=True)
                masks = masks.to(device, non_blocking=True)
                images = prep_input(images)

                with torch.amp.autocast('cuda', enabled=use_amp):
                    outputs = model(images)
                outputs_f = outputs.float()

                loss = bce_criterion(outputs_f, masks) + dice_criterion(outputs_f, masks)
                _, r, f1 = calculate_precision_recall_f1(outputs_f, masks)

                val_loss += loss.item()
                val_iou += calculate_iou(outputs_f, masks)
                val_recall += r
                val_f1 += f1

        n_val = len(val_loader)
        epoch_val_loss = val_loss / n_val
        epoch_val_iou = val_iou / n_val
        epoch_val_recall = val_recall / n_val
        epoch_val_f1 = val_f1 / n_val

        scheduler.step(epoch_val_loss)

        history_train_loss.append(epoch_train_loss)
        history_train_iou.append(epoch_train_iou)
        history_train_recall.append(epoch_train_recall)
        history_train_f1.append(epoch_train_f1)
        history_val_loss.append(epoch_val_loss)
        history_val_iou.append(epoch_val_iou)
        history_val_recall.append(epoch_val_recall)
        history_val_f1.append(epoch_val_f1)

        print(f"[Epoch {epoch+1:02d}/{epochs}] loss:{epoch_train_loss:.4f} iou:{epoch_train_iou:.4f} f1:{epoch_train_f1:.4f} | "
              f"val_loss:{epoch_val_loss:.4f} val_iou:{epoch_val_iou:.4f} val_f1:{epoch_val_f1:.4f}")

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict() if use_amp else None,
            "val_loss": epoch_val_loss,
            "history_train_loss": history_train_loss,
            "history_train_iou": history_train_iou,
            "history_train_recall": history_train_recall,
            "history_train_f1": history_train_f1,
            "history_val_loss": history_val_loss,
            "history_val_iou": history_val_iou,
            "history_val_recall": history_val_recall,
            "history_val_f1": history_val_f1,
            "best_val_loss": best_val_loss,
            "best_epoch": best_epoch,
            "arch": arch,
            "encoder_name": encoder_name,
            "encoder_weights": encoder_weights,
            "use_normalize": use_normalize,
        }, latest_ckpt_path)

        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_epoch = epoch + 1
            torch.save({"epoch": epoch + 1, "model_state_dict": model.state_dict(),
                        "val_loss": epoch_val_loss, "val_iou": epoch_val_iou, "val_f1": epoch_val_f1,
                        "arch": arch,
                        "encoder_name": encoder_name, "encoder_weights": encoder_weights,
                        "use_normalize": use_normalize},
                       best_ckpt_path)
            print(f"   💾 val_loss ดีขึ้น ที่ epoch {epoch+1} -> เซฟ best model")

        with open(csv_path, "a", newline="") as f:
            csv.writer(f).writerow([epoch + 1, epoch_train_loss, epoch_train_iou,
                                    epoch_train_recall, epoch_train_f1,
                                    epoch_val_loss, epoch_val_iou,
                                    epoch_val_recall, epoch_val_f1])

        save_plot(history_train_loss, history_val_loss,
                  history_train_iou, history_val_iou,
                  history_train_recall, history_val_recall,
                  history_train_f1, history_val_f1,
                  graph_path)

    torch.save(model.state_dict(), final_model_path)

    print(f"\n🎉 เทรนจบแล้ว! 🏆 Best model อยู่ที่ epoch {best_epoch} (val_loss={best_val_loss:.4f})")
    print(f"📁 best model:  {best_ckpt_path}")
    print(f"📁 โมเดลตัวจบ:  {final_model_path}")
    print(f"📁 metrics:     {csv_path}")


## 🔧 CONFIG — แก้ตรงนี้อย่างเดียว

- **สถาปัตยกรรม & โมเดล:** เลือกระหว่าง `ARCH = "DeepLabV3Plus"` หรือ `"Unet"`
- **Encoder:** `"resnet50"`, `"resnet34"`
- **Weights:** `"ssl"` (สำหรับ resnet50) หรือ `"imagenet"`
- **VRAM & Batch size:** แนะนำ `BATCH_SIZE = 16` สำหรับ DeepLabV3+ ResNet50
- **เมตริก:** เน้นวัด IoU, Precision, Recall, F1


In [ ]:
# ==================== การตั้งค่าโมเดล DeepLabV3+ SSL ====================
ARCH = "DeepLabV3Plus"           # โครงข่ายหลัก DeepLabV3+
ENCODER_NAME = "resnet50"        # Backbone ResNet-50
ENCODER_WEIGHTS = "ssl"          # Semi-Supervised Learning (MoCo v2 weights)
EPOCHS = 30
BATCH_SIZE = 16                  # Batch size 16 เหมาะสมกับ ResNet-50 บน GPU
LEARNING_RATE = 1e-4
DATASET_ROOT = "/home/jovyan/shared/donut/CarDashCam/CULane"

NUM_WORKERS = 4                  # จำนวน worker โหลดข้อมูล
USE_AMP = True                   # Mixed Precision (FP16) เร็วและประหยัด VRAM
USE_NORMALIZE = True             # Normalization ตาม MoCo v2
USE_CHANNELS_LAST = True         # เร่งความเร็วการคำนวณบน GPU

# --- จำนวนภาพที่ใช้เทรน ---
# ตั้งเป็น None ทั้งคู่เพื่อเทรนเต็มรูปแบบ (Train 61,182 ภาพ / Val 12,877 ภาพ)
# หรือกำหนดตัวเลข (เช่น LIMIT_TRAIN = 2000, LIMIT_VAL = 400) เพื่อทดสอบด่วน
LIMIT_TRAIN = None               # None = เต็ม 61,182 ภาพ
LIMIT_VAL = None                 # None = เต็ม 12,877 ภาพ
# ========================================================================

ARCH_PREFIX = "deeplabv3plus_" if ARCH.lower() in ("deeplabv3plus", "deeplabv3+") else ""
EXP_NAME = f"{ARCH_PREFIX}{ENCODER_NAME}_{ENCODER_WEIGHTS}_bs{BATCH_SIZE}_ep{EPOCHS}"
if LIMIT_TRAIN:
    EXP_NAME += f"_n{LIMIT_TRAIN}"

print(f"🚀 ตั้งค่าโมเดล: {ARCH} + {ENCODER_NAME} ({ENCODER_WEIGHTS})")
print(f"   Epochs: {EPOCHS} | Batch Size: {BATCH_SIZE} | LR: {LEARNING_RATE}")
if LIMIT_TRAIN or LIMIT_VAL:
    print(f"✂️  จำกัดจำนวนรูป: train={LIMIT_TRAIN}, val={LIMIT_VAL}")
else:
    print(f"📊 ชุดข้อมูลเต็ม: train=61,182 รูป, val=12,877 รูป")
print(f"🏷️  EXP_NAME = {EXP_NAME}")


## เช็คความพร้อมก่อนเทรนจริง (แนะนำให้รัน)

เซลล์นี้ใช้เวลาไม่ถึงนาที เอาไว้เช็คว่า
1. โหลด weights `imagenet` ของ resnet34 ได้ (ครั้งแรกจะดาวน์โหลด)
2. ขนาด output ถูกต้อง `(N, 1, 448, 800)`
3. **loss ไม่เป็น NaN/inf ตอนใช้ AMP** — เช็คว่า `DiceLoss` upcast ทำงานจริง

ถ้าเซลล์นี้ผ่าน แปลว่าไม่ต้องเสียเวลาเทรนไปเจอปัญหากลางทาง

In [ ]:
def smoke_test(arch=ARCH, encoder_name=ENCODER_NAME, encoder_weights=ENCODER_WEIGHTS, use_amp=USE_AMP):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = use_amp and device.type == "cuda"
    channels_last = USE_CHANNELS_LAST and device.type == "cuda"

    arch_fn = getattr(smp, arch, None)
    if arch_fn is None:
        raise ValueError(f"ไม่รู้จัก architecture: {arch} ใน segmentation_models_pytorch")
    m = arch_fn(encoder_name=encoder_name, encoder_weights=encoder_weights,
                in_channels=3, classes=1).to(device)
    if channels_last:
        m = m.to(memory_format=torch.channels_last)

    prep_input = make_input_prep(encoder_name, encoder_weights, device,
                                 normalize=USE_NORMALIZE, channels_last=channels_last)

    h, w = IMG_SIZE
    images = torch.rand(2, 3, h, w, device=device)          # เลียนแบบ output ของ dataset ช่วง [0,1]
    masks = (torch.rand(2, 1, h, w, device=device) > 0.9).float()

    with torch.amp.autocast('cuda', enabled=use_amp):
        out = m(prep_input(images))
    out_f = out.float()

    loss = nn.BCEWithLogitsLoss()(out_f, masks) + DiceLoss()(out_f, masks)
    iou = calculate_iou(out_f, masks)

    print(f"สถาปัตยกรรม: {arch} + {encoder_name} ({encoder_weights})")
    print(f"output shape: {tuple(out.shape)} (คาดหวัง (2, 1, {h}, {w}))")
    print(f"output dtype ใน autocast: {out.dtype}")
    print(f"loss = {loss.item():.4f} | IoU = {iou:.4f}")

    assert tuple(out.shape) == (2, 1, h, w), "ขนาด output ไม่ถูกต้อง"
    assert torch.isfinite(loss), "loss เป็น NaN/inf — DiceLoss upcast มีปัญหา"
    print("✅ ผ่านทุกข้อ พร้อมเทรนจริง")

    del m, images, masks, out, out_f
    if device.type == "cuda":
        torch.cuda.empty_cache()


smoke_test()


## รันเทรนจริง — รอจนเห็น 🎉 เทรนจบแล้ว

เวลาที่ใช้ขึ้นกับ `LIMIT_TRAIN` (วัดจากเครื่องนี้จริง 0.366 วิ/step ที่ bs32):

| `LIMIT_TRAIN` | steps/epoch | 30 epochs |
| --- | --- | --- |
| 1,000 | 32 | ~7 นาที |
| **2,000** | 63 | **~13–15 นาที** |
| 5,000 | 157 | ~30 นาที |
| `None` (61,182 เต็ม) | 1,912 | **~6 ชั่วโมง** |

ปิดแท็บหรือปิดคอมได้ kernel รันต่อบนเซิร์ฟเวอร์ ระหว่างนี้ `results/metrics_*.csv` กับกราฟ
จะถูกอัปเดต **ทุก epoch** เข้ามาดูความคืบหน้าได้เรื่อยๆ ไม่ต้องรอจบ

ถ้า kernel ตายกลางทาง รันเซลล์นี้ซ้ำแล้วมัน resume ต่อจาก epoch เดิม
แต่ถ้าเทรนครบแล้วมันจะ **ข้ามเงียบๆ** ต้องลบ `models/final_result_epoch_{exp_name}.pth` ถ้าอยากเทรนใหม่

In [ ]:
train(
    arch=ARCH,
    encoder_name=ENCODER_NAME,
    encoder_weights=ENCODER_WEIGHTS,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    dataset_root=DATASET_ROOT,
    num_workers=NUM_WORKERS,
    use_amp=USE_AMP,
    use_normalize=USE_NORMALIZE,
    use_channels_last=USE_CHANNELS_LAST,
    limit_train=LIMIT_TRAIN,
    limit_val=LIMIT_VAL,
)


## วัดผลบนชุด test

ใช้ `test_list.csv` (13,298 รูป) ซึ่งเป็นชุดที่โมเดล **ไม่เคยเห็น** ระหว่างเทรน
(split ถูกแบ่งตาม `clip_id` ไว้แล้ว จึงไม่มีเฟรมจากคลิปเดียวกันรั่วข้ามชุด)

เลือกโมเดลที่จะวัดจาก `EVAL_TARGETS` โดยแก้ค่า `EVAL_CHOICE` อย่างเดียว

**สำคัญ:** แต่ละโมเดลจับคู่ค่า normalize ที่ถูกต้องไว้ให้แล้ว เพราะ**ใส่ผิดแล้วผลเพี้ยนหนัก**
โมเดลชุดเก่าเทรนมาด้วย input `[0,1]` แบบไม่ normalize ถ้าเผลอใส่ ImageNet normalization เข้าไป
คะแนนจะร่วงทันที (วัดจริงกับ `unet_lane_model_bs32_ep30.pth` ได้ F1 0.667 → 0.414)
จึงทำเป็นตารางจับคู่ไว้ ไม่ต้องมาจำเอง

| `EVAL_CHOICE` | ไฟล์ | normalize |
| --- | --- | --- |
| `new_final` | `unet_lane_model_{exp_name}.pth` — รันใหม่ epoch สุดท้าย (ค่าเริ่มต้น) | ✅ |
| `new_best` | `best_model_{exp_name}.pth` — รันใหม่ ตัว val_loss ต่ำสุด | ✅ |
| `old_bs32_ep30` | `unet_lane_model_bs32_ep30.pth` — resnet34 เดิม 30 epochs | ❌ |
| `old_bs32_ep20` | `best_unet_lane_model_bs32_ep20.pth` — resnet34 เดิม 20 epochs | ❌ |

⚠️ `new_final` คือ epoch สุดท้าย **ไม่ใช่** ตัวที่ `val_loss` ต่ำสุด จากผลการทดลองเก่าในโปรเจกต์นี้
`val_loss` มักเริ่มสูงขึ้นช่วงท้าย (เช่น `resnet50_ssl` จบที่ 0.443 ขณะที่ train F1 พุ่งถึง 0.94)
ดังนั้นอาจได้คะแนนต่ำกว่า `new_best` — วัดทั้งสองตัวแล้วเทียบกันได้เลย

**ตัวเลือก `old_*` รันได้เลยไม่ต้องรอเทรน** ใช้เป็น baseline เทียบกับรันใหม่ได้ทันที
ผลจะถูกบันทึกแยกไฟล์ตามชื่อโมเดล ไม่ทับกัน

In [ ]:
def load_seg_model(model_path, encoder_name, device, channels_last=False):
    """
    โหลด weights แล้วสร้างโมเดลให้ตรงสถาปัตยกรรมเอง

    ทำไมต้องเดาเอง: โปรเจกต์นี้มีทั้ง U-Net และ DeepLabv3+ ปนกันใน models/
    ถ้าสร้างผิดคลาส load_state_dict จะ error ทันที (ดีกว่าเงียบ) แต่ก็เสียเวลา
    ดูจาก key ในไฟล์แทนการเชื่อชื่อไฟล์ เพราะชื่อไฟล์ตั้งมือ พลาดได้
    """
    obj = torch.load(model_path, map_location="cpu", weights_only=False)
    state = obj["model_state_dict"] if isinstance(obj, dict) and "model_state_dict" in obj else obj
    keys = list(state.keys())

    if any("decoder.aspp" in k for k in keys):
        arch = "DeepLabV3Plus"
    elif any("decoder.blocks" in k for k in keys):
        arch = "Unet"
    else:
        raise ValueError(f"ไม่รู้จักสถาปัตยกรรมของ {model_path} — ไม่เจอทั้ง decoder.aspp และ decoder.blocks")

    model = getattr(smp, arch)(encoder_name=encoder_name, encoder_weights=None,
                               in_channels=3, classes=1)
    model.load_state_dict(state)          # ปล่อยให้ error ถ้า encoder ไม่ตรง อย่า strict=False
    model.to(device).eval()
    if channels_last:
        model = model.to(memory_format=torch.channels_last)
    print(f"   สถาปัตยกรรมที่ตรวจได้: {arch} + {encoder_name}")
    return model


# ==================== เลือกโมเดลที่จะวัด ====================
# (path, ต้อง normalize หรือไม่, encoder)
# โมเดลชุด old_* เทรนมาด้วย input [0,1] แบบไม่ normalize จึงต้องเป็น False
FINAL_NAME = f"models/{'unet_lane_model' if ARCH == 'Unet' else 'final_model'}_{EXP_NAME}.pth"
EVAL_TARGETS = {
    "new_final":     (FINAL_NAME,                                  True,  ENCODER_NAME),
    "new_best":      (f"models/best_model_{EXP_NAME}.pth",         True,  ENCODER_NAME),
    "old_bs32_ep30": ("models/unet_lane_model_bs32_ep30.pth",      False, "resnet34"),
    "old_bs32_ep20": ("models/best_unet_lane_model_bs32_ep20.pth", False, "resnet34"),
}

EVAL_CHOICE = "new_final"        # <-- แก้ตรงนี้อย่างเดียว
# ================================================================

MODEL_PATH, EVAL_NORMALIZE, EVAL_ENCODER = EVAL_TARGETS[EVAL_CHOICE]

# ตั้งชื่อไฟล์ผลลัพธ์ตามโมเดลที่วัด เพื่อไม่ให้ผลของแต่ละโมเดลทับกัน
# (new_final กับ new_best ต้องแยกไฟล์ด้วย ไม่งั้นวัดตัวที่สองแล้วทับผลตัวแรก เทียบกันไม่ได้)
if EVAL_CHOICE == "new_final":
    EVAL_TAG = EXP_NAME
elif EVAL_CHOICE == "new_best":
    EVAL_TAG = f"{EXP_NAME}_best"
else:
    EVAL_TAG = os.path.splitext(os.path.basename(MODEL_PATH))[0]

print(f"จะวัด: {EVAL_CHOICE} -> {MODEL_PATH}")
print(f"  encoder={EVAL_ENCODER} | normalize={EVAL_NORMALIZE} | ผลจะเซฟเป็น test_metrics_{EVAL_TAG}.csv")


def evaluate_test(model_path, encoder_name, batch_size, dataset_root,
                  use_normalize=True, num_workers=4, use_amp=True,
                  use_channels_last=True, threshold=0.5, exp_name=None, save_csv=True):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = use_amp and device.type == "cuda"
    use_channels_last = use_channels_last and device.type == "cuda"

    if not os.path.exists(model_path):
        print(f"❌ ไม่พบไฟล์โมเดล: {model_path}")
        print("   ต้องเทรนให้จบก่อน หรือเปลี่ยน MODEL_PATH ให้ชี้ไฟล์ที่มีอยู่")
        return None

    img_dir = os.path.join(dataset_root, "images")
    mask_dir = os.path.join(dataset_root, "masks")
    test_csv = os.path.join(dataset_root, "test_list.csv")
    if not os.path.exists(test_csv):
        print(f"❌ ไม่พบ {test_csv}")
        return None

    test_dataset = LaneDatasetFromCSV(test_csv, img_dir, mask_dir, augment=False)
    loader_kwargs = dict(num_workers=num_workers, pin_memory=(device.type == "cuda"))
    if num_workers > 0:
        loader_kwargs.update(prefetch_factor=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, **loader_kwargs)

    print(f"🧪 Test: {len(test_dataset)} รูป {test_dataset.category_counts()}")
    print(f"📦 โมเดล: {model_path}")

    # weights=None เพราะน้ำหนักมาจากไฟล์ ไม่ต้องดาวน์โหลด imagenet ใหม่
    model = load_seg_model(model_path, encoder_name, device)
    model.to(device).eval()
    if use_channels_last:
        model = model.to(memory_format=torch.channels_last)

    prep_input = make_input_prep(encoder_name, "imagenet", device,
                                 normalize=use_normalize, channels_last=use_channels_last)

    iou = precision = recall = f1 = 0.0
    with torch.inference_mode():
        for images, masks in tqdm(test_loader, desc="[Test]", leave=False):
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            images = prep_input(images)

            with torch.amp.autocast('cuda', enabled=use_amp):
                outputs = model(images)
            outputs_f = outputs.float()

            p, r, f = calculate_precision_recall_f1(outputs_f, masks, threshold=threshold)
            iou += calculate_iou(outputs_f, masks, threshold=threshold)
            precision += p
            recall += r
            f1 += f

    n = len(test_loader)
    # เฉลี่ยแบบต่อ batch เหมือนที่โค้ดเดิมทำ เพื่อให้เทียบกับ metrics_*.csv เก่าได้
    results = {
        "iou": iou / n,
        "precision": precision / n,
        "recall": recall / n,
        "f1_score": f1 / n,
    }

    print("\n📊 ผลบนชุด test")
    print(f"   IoU (accuracy) : {results['iou']:.4f}")
    print(f"   Precision      : {results['precision']:.4f}")
    print(f"   Recall         : {results['recall']:.4f}")
    print(f"   F1-score       : {results['f1_score']:.4f}")

    if save_csv and exp_name:
        os.makedirs("results/test", exist_ok=True)
        out_csv = f"results/test/test_metrics_{exp_name}.csv"
        with open(out_csv, "w", newline="") as f_out:
            w = csv.writer(f_out)
            w.writerow(["model_path", "n_images", "threshold", "normalized",
                        "test_iou", "test_precision", "test_recall", "test_f1_score"])
            w.writerow([model_path, len(test_dataset), threshold, use_normalize,
                        results["iou"], results["precision"],
                        results["recall"], results["f1_score"]])
        print(f"\n📁 บันทึกผลไว้ที่: {out_csv}")

    return results


test_results = evaluate_test(
    model_path=MODEL_PATH,
    encoder_name=EVAL_ENCODER,
    batch_size=BATCH_SIZE,
    dataset_root=DATASET_ROOT,
    use_normalize=EVAL_NORMALIZE,
    num_workers=NUM_WORKERS,
    use_amp=USE_AMP,
    use_channels_last=USE_CHANNELS_LAST,
    exp_name=EVAL_TAG,
)


## 👁️ ดูผลลัพธ์ด้วยตา — ภาพ Overlay หน้ากากเลน (Ground Truth vs Prediction)

แสดงตัวอย่างภาพทดสอบพร้อมหน้ากากเลนที่โมเดลทำนาย:
- **สีเขียว:** Ground Truth (เฉลยจริง)
- **สีแดง:** Prediction (โมเดลทำนาย)
- **สีเหลือง:** Overlap (จุดที่ทำนายทับซ้อนกับเฉลยได้ถูกต้อง)
- แสดงค่า IoU ของแต่ละภาพบนหัวข้อ


In [ ]:
PLOT_N    = 12       # จำนวนภาพ
PLOT_SEED = 7        # seed สำหรับสุ่ม


def plot_prediction_samples(model_path, encoder_name, dataset_root, use_normalize=True,
                            n_samples=PLOT_N, seed=PLOT_SEED, threshold=0.5,
                            tag=None, use_amp=True):
    import random
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = use_amp and device.type == "cuda"

    if not os.path.exists(model_path):
        print(f"❌ ไม่พบไฟล์โมเดล: {model_path}")
        return None

    img_dir = os.path.join(dataset_root, "images")
    mask_dir = os.path.join(dataset_root, "masks")
    ds = LaneDatasetFromCSV(os.path.join(dataset_root, "test_list.csv"), img_dir, mask_dir,
                            augment=False)

    bycat = {}
    for i, (_, c) in enumerate(ds.samples):
        bycat.setdefault(c, []).append(i)
    rng = random.Random(seed)
    cats_sorted = sorted(bycat)
    base, extra = divmod(n_samples, len(cats_sorted))
    picks = []
    for k, c in enumerate(cats_sorted):
        want = base + (1 if k < extra else 0)
        if want:
            picks += [(i, c) for i in rng.sample(bycat[c], min(want, len(bycat[c])))]
    picks = picks[:n_samples]

    model = load_seg_model(model_path, encoder_name, device)
    prep_input = make_input_prep(encoder_name, "imagenet", device,
                                 normalize=use_normalize, channels_last=False)

    cols = 3
    rows_n = int(np.ceil(len(picks) / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(6.2 * cols, 4.45 * rows_n))
    axes = np.atleast_1d(axes).ravel()

    for ax, (idx, cat) in zip(axes, picks):
        image_t, mask_t = ds[idx]
        with torch.inference_mode():
            x = image_t.unsqueeze(0).to(device)
            with torch.amp.autocast('cuda', enabled=use_amp):
                logit = model(prep_input(x))
            prob = torch.sigmoid(logit.float())[0, 0].cpu().numpy()

        rgb = np.ascontiguousarray((image_t.permute(1, 2, 0).numpy() * 255).astype(np.uint8))
        gt = mask_t[0].numpy() > 0.5
        pd = prob > threshold

        intersection = np.logical_and(gt, pd).sum()
        union = np.logical_or(gt, pd).sum()
        sample_iou = (intersection / union) if union > 0 else (1.0 if not gt.any() and not pd.any() else 0.0)

        vis = (rgb * 0.5).astype(np.uint8)
        vis[gt] = (0.35 * vis[gt] + 0.65 * np.array([0, 230, 0])).astype(np.uint8)
        vis[pd] = (0.35 * vis[pd] + 0.65 * np.array([255, 40, 40])).astype(np.uint8)
        vis[gt & pd] = (0.3 * vis[gt & pd] + 0.7 * np.array([255, 230, 0])).astype(np.uint8)

        ax.imshow(vis)
        ax.set_title(f"[{cat}] IoU: {sample_iou:.3f}", fontsize=11, fontweight='bold')
        ax.axis("off")

    for ax in axes[len(picks):]:
        ax.axis("off")

    fig.suptitle("Green = Ground Truth   |   Red = Prediction   |   Yellow = Overlap\n"
                 f"Model: {os.path.basename(model_path)} (normalize={use_normalize})",
                 fontsize=12, fontweight='bold')
    plt.tight_layout(rect=(0, 0, 1, 0.96), h_pad=2.0)

    os.makedirs("results/evaluation", exist_ok=True)
    out = f"results/evaluation/prediction_samples_{tag or 'test'}.png"
    plt.savefig(out, dpi=110)
    print(f"📁 บันทึกภาพไว้ที่: {out}")
    plt.show()


plot_prediction_samples(
    model_path=MODEL_PATH,
    encoder_name=EVAL_ENCODER,
    dataset_root=DATASET_ROOT,
    use_normalize=EVAL_NORMALIZE,
    tag=EVAL_TAG,
)
